# NB02 – Data Preparation

## Purpose

The data collected in NB01 is stored exactly as it was returned by the Open Brewery DB API. While preserving the raw data is important for reproducibility, it is not suitable for analysis because it contains variables that are not relevant to this project, inconsistent formatting, and missing values.

The purpose of this notebook is to create a clean dataset that can be used directly in NB03. Keeping all preparation steps in one notebook makes the workflow easier to follow and ensures every change made to the original data is documented before any analysis begins.

## Load the Raw Dataset

The first step is to load the raw JSON file created in NB01 and convert it into a pandas DataFrame. Inspecting the original data before making any modifications helps identify potential quality issues and provides a clear starting point for the cleaning process.

In [2]:
import json
import pandas as pd
from pathlib import Path

In [3]:
raw_file = Path("../data/raw/breweries_raw.json")

with open(raw_file, "r") as f:
    breweries = json.load(f)

breweries_df = pd.DataFrame(breweries)

print(f"Rows: {len(breweries_df)}")
print(f"Columns: {len(breweries_df.columns)}")

Rows: 2000
Columns: 16


### Initial Inspection

Loading the complete dataset confirms that the data collection stage was successful and provides an opportunity to inspect the structure of the returned records. Before making any changes, it is important to understand how the variables are organised and whether the dataset contains information that will be useful for answering the research question.

In [4]:
breweries_df.head()

,id,name,brewery_type,address_1,address_2,address_3,city,state_province,postal_code,country,longitude,latitude,phone,website_url,state,street
0,ae7b3174-8be8-4d53-a3a5-9b8240970eea,'s,brewpub,1 Friesener Straße,NaN,NaN,Kronach,Bayern,96317,Germany,11.327765,50.241246,+49 9261 628000,http://www.antla.de,Bayern,1 Friesener Straße
1,aa7cbe9b-3a0f-4888-9884-6186b0042b55,’t Drankorgel,micro,31a Geelsebaan,NaN,NaN,Mol,Vlaanderen,2400,Belgium,5.056955,51.176258,+32 496 76 40 63,http://tdrankorgel.be/,Vlaanderen,31a Geelsebaan
2,1034754f-abf9-4b42-bd6d-37e110ba4b1a,'T Kroontje,micro,62 Hogebrug,NaN,NaN,Lebbeke,Vlaanderen,9280,Belgium,4.091393,51.007888,+32 495 43 33 25,http://www.tkroontje.be/,Vlaanderen,62 Hogebrug
3,5128df48-79fc-4f0f-8b52-d06be54d0cec,(405) Brewing Co,micro,1716 Topeka St,NaN,NaN,Norman,Oklahoma,73069-8224,United States,-97.468182,35.257389,4058160490,http://www.405brewing.com,Oklahoma,1716 Topeka St
4,9c5a66c8-cc13-416f-a5d9-0a769c87d318,(512) Brewing Co,micro,407 Radam Ln Ste F200,NaN,NaN,Austin,Texas,78745-1197,United States,NaN,NaN,5129211545,http://www.512brewing.com,Texas,407 Radam Ln Ste F200


### Discussion

The preview shows that each brewery record contains a mixture of identifying information, location details, and descriptive variables such as brewery type. It also suggests that not every variable will be useful for this project. The next step is to examine the dataset in more detail so that cleaning decisions are based on evidence rather than assumptions.

## Examine the Dataset Structure

Before deciding how the data should be cleaned, it is necessary to understand the variables returned by the API and assess their quality. Inspecting the dataset at this stage helps identify missing values, inconsistent formatting, and variables that may not contribute to answering the research question.

Rather than removing or modifying data immediately, this inspection provides evidence for each transformation that will be performed later in the notebook.

In [5]:
breweries_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              2000 non-null   str    
 1   name            2000 non-null   str    
 2   brewery_type    2000 non-null   str    
 3   address_1       1882 non-null   str    
 4   address_2       43 non-null     str    
 5   address_3       6 non-null      str    
 6   city            2000 non-null   str    
 7   state_province  2000 non-null   str    
 8   postal_code     2000 non-null   str    
 9   country         2000 non-null   str    
 10  longitude       1622 non-null   float64
 11  latitude        1622 non-null   float64
 12  phone           1790 non-null   str    
 13  website_url     1766 non-null   str    
 14  state           2000 non-null   str    
 15  street          1882 non-null   str    
dtypes: float64(2), str(14)
memory usage: 250.1 KB


### Discussion

The dataset summary provides information about each variable, including its data type and the number of non-null observations. Comparing these values makes it possible to identify columns that contain missing information and determine whether each variable is suitable for analysis.

The next step is to quantify the missing values so that cleaning decisions are based on measurable evidence rather than visual inspection alone.

In [6]:
missing_values = breweries_df.isnull().sum().sort_values(ascending=False)

missing_values

address_3         1994
address_2         1957
longitude          378
latitude           378
website_url        234
phone              210
address_1          118
street             118
state_province       0
city                 0
name                 0
brewery_type         0
id                   0
country              0
postal_code          0
state                0
dtype: int64

In [7]:
missing_values[missing_values > 0]

address_3      1994
address_2      1957
longitude       378
latitude        378
website_url     234
phone           210
address_1       118
street          118
dtype: int64

### Discussion

The inspection shows that most of the core variables required for this project are complete. Information such as the brewery name, brewery type, city, state, postal code, and unique identifier contains no missing values, making these variables reliable for analysing geographic patterns across the United States.

Missing values are concentrated in a small number of fields. Nearly every record is missing `address_2` and `address_3`, suggesting these fields are optional rather than incomplete. The `website_url` and `phone` variables also contain missing values, while latitude and longitude are unavailable for 378 breweries. These missing geographic coordinates will need to be considered carefully because they may affect any location-based visualisations produced later in the project.

Rather than removing records immediately, the next stage is to determine which variables contribute to answering the research question. Columns that provide little analytical value or are almost entirely empty may be removed, while important variables with missing values will be handled using an appropriate cleaning strategy.

## Evaluate Variables

Not every variable returned by the API is necessary for answering the research question. Before cleaning the dataset, each column should be evaluated according to whether it provides useful information for analysing brewery characteristics across the United States.

Reviewing the available variables before removing any data helps ensure that important information is not discarded without justification.

In [8]:
breweries_df.columns.tolist()

['id',
 'name',
 'brewery_type',
 'address_1',
 'address_2',
 'address_3',
 'city',
 'state_province',
 'postal_code',
 'country',
 'longitude',
 'latitude',
 'phone',
 'website_url',
 'state',
 'street']

### Selecting Relevant Variables

The inspection suggests that most variables provide information that could contribute to understanding brewery locations or characteristics. However, `address_2` and `address_3` contain missing values for almost every brewery, indicating that these fields are optional and provide little additional information for this project.

Rather than removing records with missing values, only these two columns will be excluded because they contribute very little analytical value while increasing the complexity of the dataset. The remaining variables will be retained until there is clear evidence that further cleaning is required.

In [9]:
breweries_df = breweries_df.drop(columns=["address_2", "address_3"])

print("Remaining columns:")
print(breweries_df.columns.tolist())

Remaining columns:
['id', 'name', 'brewery_type', 'address_1', 'city', 'state_province', 'postal_code', 'country', 'longitude', 'latitude', 'phone', 'website_url', 'state', 'street']


### Discussion

Removing only the two nearly empty address fields simplifies the dataset without discarding meaningful information. All variables that may contribute to the analysis have been retained at this stage because it is generally preferable to postpone additional reductions until there is a clear analytical reason to do so.

The next step is to check whether any duplicate brewery records exist before continuing with further data preparation.

## Identify Duplicate Records

Before analysing the dataset, it is important to confirm that each brewery appears only once. Duplicate records can distort summary statistics, maps, and visualisations by giving some breweries more influence than others.

Checking for duplicates before making further changes helps ensure that any patterns observed later are based on unique brewery records rather than repeated observations.

In [10]:
duplicate_count = breweries_df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


### Discussion

No duplicate records were identified in the dataset. This suggests that each brewery is represented by a single observation, so no duplicate removal is required.

Verifying this before continuing is still an important quality check because it confirms that later analyses will not be influenced by repeated records.

## Review Data Types

The structure of each variable influences how it can be analysed. Reviewing the data types at this stage helps confirm that numerical information has been imported correctly and that categorical variables are stored in a suitable format before further cleaning begins.

Any inconsistencies identified here can be corrected before the processed dataset is created.

In [11]:
breweries_df.dtypes

id                    str
name                  str
brewery_type          str
address_1             str
city                  str
state_province        str
postal_code           str
country               str
longitude         float64
latitude          float64
phone                 str
website_url           str
state                 str
street                str
dtype: object

In [12]:
breweries_df.describe(include="all")

,id,name,brewery_type,address_1,city,state_province,postal_code,country,longitude,latitude,phone,website_url,state,street
count,2000,2000,2000,1882,2000,2000,2000,2000,1622.000000,1622.000000,1790,1766,2000,1882
unique,2000,1967,13,1841,1363,129,1904,18,NaN,NaN,1726,1688,129,1841
top,ae7b3174-8be8-4d53-a3a5-9b8240970eea,Ballast Point Brewing Company,micro,20 Bond Street,San Diego,California,1000,United States,NaN,NaN,5415851007,http://www.abcbrew.com,California,20 Bond Street
freq,1,6,976,3,22,159,11,1380,NaN,NaN,3,5,159,3
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-49.749411,36.450223,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76.303667,22.520991,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-157.939931,-45.717790,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-105.069560,36.009784,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-80.962078,41.962375,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.038770,48.522385,NaN,NaN,NaN,NaN


### Discussion

The data types are consistent with the information stored in each variable. Text-based fields such as brewery names, locations, and contact information have been imported as strings, while latitude and longitude have been recognised as numerical variables. This means the geographic coordinates can be used directly for mapping and other quantitative analyses without requiring additional type conversion.

The summary statistics also provide a better understanding of the dataset. There are 2,000 brewery records representing 13 different brewery types, while the longitude and latitude variables contain 1,622 valid observations. This confirms that some breweries cannot be included in location-based visualisations unless missing coordinates are addressed. The inspection therefore provides a clear basis for the remaining cleaning decisions rather than making assumptions about the quality of the data.

## Handle Missing Values

The inspection identified missing values in several variables, but not every missing value should be treated in the same way. The appropriate strategy depends on how each variable contributes to the research question.

Variables that are not essential for the planned analysis can be retained with missing values, while observations missing information required for later visualisations may need to be excluded. Making these decisions at this stage helps preserve as much information as possible while ensuring that the processed dataset is suitable for analysis.

In [13]:
breweries_df[["latitude", "longitude"]].isnull().sum()

latitude     378
longitude    378
dtype: int64

In [14]:
missing_coordinates = breweries_df[
    breweries_df["latitude"].isna() | breweries_df["longitude"].isna()
]

len(missing_coordinates)

378

### Discussion

The inspection confirms that both the `latitude` and `longitude` variables contain 378 missing values. Because these two variables are missing together, the affected breweries cannot be plotted accurately on a map. However, they still contain valid information about brewery type, city, and state, meaning they remain useful for several parts of the planned analysis.

Removing these breweries from the entire dataset would unnecessarily reduce the amount of available information. Instead, the complete dataset will be preserved for analyses that do not require geographic coordinates, while a separate dataset containing only breweries with valid coordinates will be created specifically for mapping. This approach retains as much information as possible while ensuring that geographic visualisations are based on complete location data.

### Preparing Data for Geographic Analysis

Missing geographic coordinates do not affect every type of analysis. For example, breweries without latitude or longitude can still be included when comparing brewery types or state distributions. However, they cannot be displayed accurately on a map.

Rather than removing these observations from the entire dataset, a separate dataset will be created for geographic visualisations. This approach preserves as much information as possible while ensuring that any maps produced later are based only on valid location data.

In [16]:
map_df = breweries_df.dropna(subset=["latitude", "longitude"]).copy()

print(f"Original dataset: {len(breweries_df)} breweries")
print(f"Mapping dataset: {len(map_df)} breweries")

Original dataset: 2000 breweries
Mapping dataset: 1622 breweries


## Save the Processed Data

The final step is to save the cleaned datasets so they can be used directly in NB03. Exporting the processed data ensures that the analysis notebook focuses entirely on exploration and visualisation rather than repeating preparation steps.

In [17]:
processed_folder = Path("../data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

breweries_df.to_csv(
    processed_folder / "breweries_clean.csv",
    index=False
)

map_df.to_csv(
    processed_folder / "breweries_map.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


# Summary

This notebook transformed the raw brewery data collected in NB01 into datasets that are ready for analysis. Throughout the preparation process, each cleaning decision was supported by evidence gathered during the initial inspection of the data.

Rather than removing large amounts of information, only variables with very limited analytical value were excluded, while breweries with missing geographic coordinates were retained in the main dataset and separated only for mapping purposes. This approach preserves as much information as possible while ensuring that the processed datasets are appropriate for the analyses performed in NB03.